In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from google.colab import files

df = pd.read_csv("data_training.csv")
X_raw = df[['temp', 'cpu']].values.astype(np.float32)
le = LabelEncoder()
le.classes_ = np.array(['STANDARD_INT8', 'PRUNED_VARIANT', 'MIXED_PRECISION'])
y = le.transform(df['label'].values)
TEMP_MEAN = float(X_raw[:, 0].mean())
TEMP_STD  = float(X_raw[:, 0].std())
CPU_MEAN  = float(X_raw[:, 1].mean())
CPU_STD   = float(X_raw[:, 1].std())
X = np.zeros_like(X_raw)
X[:, 0] = (X_raw[:, 0] - TEMP_MEAN) / TEMP_STD
X[:, 1] = (X_raw[:, 1] - CPU_MEAN)  / CPU_STD
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3,  activation='softmax')
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=0,
          validation_data=(X_test, y_test),
          callbacks=[tf.keras.callbacks.EarlyStopping(
              patience=10, restore_best_weights=True)])
_, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy: {acc*100:.1f}%")
print(f"TEMP_MEAN={TEMP_MEAN:.4f} TEMP_STD={TEMP_STD:.4f}")
print(f"CPU_MEAN={CPU_MEAN:.4f}  CPU_STD={CPU_STD:.4f}")

model.save('/content/model.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
with open('/content/model.tflite', 'wb') as f:
    f.write(tflite_bytes)

files.download('/content/model.keras')
files.download('/content/model.tflite')
print("Done!")